# Import log files into panda dataframes

In [2]:
import pandas as pd
from glob import glob
import os


# List all .dat files in the current directory
# elephant
dat_file_root = "/srv/data/stratbox_simulations/stratbox_particle_runs/bx5/smd132/sn34/pe300/4pc_resume/4pc"

dat_files = glob(os.path.join(dat_file_root, "SNfeedback.dat"))

# Initialize an empty DataFrame
all_data = pd.DataFrame()

# Read and concatenate data from all .dat files
for dat_file in dat_files:
    # Assuming space-separated values in the .dat files
    df = pd.read_csv(dat_file, delim_whitespace=True, header=None,
                     names=['n_SN', 'type', 'n_timestep', 'n_tracer', 'time',
                            'posx', 'posy', 'posz', 'radius', 'mass'])
    
    # Convert the columns to numerical
    df = df.iloc[1:]
    df['n_SN'] = df['n_SN'].map(int)
    df['type'] = df['type'].map(int)
    df['n_timestep'] = df['n_timestep'].map(int)
    df['n_tracer'] = df['n_tracer'].map(int)
    df['time'] = pd.to_numeric(df['time'],errors='coerce')
    df['posx'] = pd.to_numeric(df['posx'],errors='coerce')
    df['posy'] = pd.to_numeric(df['posy'],errors='coerce')
    df['posz'] = pd.to_numeric(df['posz'],errors='coerce')
    df['radius'] = pd.to_numeric(df['radius'],errors='coerce')
    df['mass'] = pd.to_numeric(df['mass'],errors='coerce')
    all_data = pd.concat([all_data, df], ignore_index=True)
    all_data = all_data.drop(df[df['n_tracer'] != 0].index)

all_data.head()


,n_SN,type,n_timestep,n_tracer,time,posx,posy,posz,radius,mass
0,1,2,12,0,2.869811e+12,9.763277e+20,1.530785e+21,-1.072755e+21,1.243486e+20,4.098446e+35
1,2,1,20,0,4.796885e+12,-1.145076e+21,-1.434358e+21,-1.205343e+19,3.511398e+19,9.042147e+35
2,3,2,26,0,5.738623e+12,3.977631e+20,-1.337930e+21,4.700837e+20,4.278450e+19,4.298436e+35
3,4,2,34,0,8.607434e+12,-1.120969e+21,1.482572e+21,-1.084808e+20,3.511398e+19,8.470546e+35
4,5,1,40,0,9.592771e+12,1.434358e+21,-8.557934e+20,2.205777e+21,3.085678e+20,4.513983e+33


In [9]:
# convert seconds to Megayears
def seconds_to_megayears(seconds):
    return seconds / (1e6 * 365 * 24 * 3600)

# Convert pixel value to pc
def pixel2pc(coord, x_y_z, top_z = 500):
    if x_y_z == "x":
        return coord - top_z
    elif x_y_z == "y":
        return top_z - coord
    elif x_y_z == "z":
        return coord - top_z
    return coord

def pix_256_2pc(pix_256):
    return pix_256 * (1000 / 256)

def pc2pix_256(pc):
    return pc * (256 / 1000)

def cm2pc(cm):
    return cm * 3.24077929e-19

# filter the DataFrame
def filter_data(df, range_coord):
    return df[(df['posx_pc'] > range_coord[0]) & (df['posx_pc'] < range_coord[0] + range_coord[2]) & 
              (df['posy_pc'] > range_coord[1]) & (df['posy_pc'] < range_coord[1] + range_coord[3]) & 
              (df['posz_pc'] > range_coord[4]) & (df['posz_pc'] < range_coord[5])]

def timestamp2Myr(timestamp):
    return (timestamp - 200) * 0.1 + 191

# Convert time to Megayears
all_data['time_Myr'] = seconds_to_megayears(all_data['time'])

# Convert 'pos' from centimeters to parsecs
all_data['posx_pc'] = cm2pc(all_data['posx'])
all_data['posy_pc'] = cm2pc(all_data['posy'])
all_data['posz_pc'] = cm2pc(all_data['posz'])

# Sort the DataFrame by time in ascending order
all_data.sort_values(by='time_Myr', inplace=True)

In [6]:
low_x0, low_y0, low_w, low_h, bottom_z, top_z = -300 , -450, 50, 50, -100, 0
# low_x0, low_y0, low_w, low_h = pixel2pc(low_x0, "x"), pixel2pc(low_y0, "y"), pixel2pc(low_w), pixel2pc(low_h)

In [4]:
all_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12250 entries, 0 to 14124
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   n_SN        12250 non-null  int64  
 1   type        12250 non-null  int64  
 2   n_timestep  12250 non-null  int64  
 3   n_tracer    12250 non-null  int64  
 4   time        12250 non-null  float64
 5   posx        12250 non-null  float64
 6   posy        12250 non-null  float64
 7   posz        12250 non-null  float64
 8   radius      12250 non-null  float64
 9   mass        12250 non-null  float64
 10  time_Myr    12250 non-null  float64
 11  posx_pc     12250 non-null  float64
 12  posy_pc     12250 non-null  float64
 13  posz_pc     12250 non-null  float64
dtypes: float64(10), int64(4)
memory usage: 1.4 MB


In [15]:
low_x0, low_y0, low_w, low_h, bottom_z, top_z = -400, -500, 100, 100, -500, 500

In [17]:
start_yr = 191
end_yr = start_yr + 15

# Filter data based on specified conditions
# filtered_data = all_data[(all_data['time_Myr'] >= start_yr) & (all_data['time_Myr'] <= end_yr)]
filtered_data = filter_data(all_data[(all_data['time_Myr'] >= start_yr) & (all_data['time_Myr'] <= end_yr)],
                            (low_x0, low_y0, low_w, low_h, bottom_z, top_z))
# filtered_data = filter_data(all_data[(all_data['time_Myr'] >= start_yr) & (all_data['time_Myr'] <= end_yr)], (-92, -101, 50, 50, -400, 400))


# Print the resulting DataFrame
filtered_data
# filtered_data.iloc[0]["posx_pc"]

,n_SN,type,n_timestep,n_tracer,time,posx,posy,posz,radius,mass,time_Myr,posx_pc,posy_pc,posz_pc
6239,6240,2,47656,0,6.041718e+15,-1.030568e+21,-1.368064e+21,-2.832556e+20,4.630299e+19,3.945900e+35,191.581618,-333.984376,-443.359380,-91.796875
6254,6255,3,47798,0,6.056860e+15,-9.823544e+20,-1.295743e+21,-3.013357e+19,7.439512e+19,3.927065e+35,192.061774,-318.359376,-419.921870,-9.765625
6300,6301,3,48261,0,6.097788e+15,-1.126995e+21,-1.428331e+21,2.591487e+20,3.953338e+19,3.955423e+35,193.359580,-365.234368,-462.890617,83.984374
6302,6303,1,48290,0,6.100372e+15,-1.018515e+21,-1.500652e+21,-9.341407e+20,4.630299e+19,4.064423e+35,193.441533,-330.078135,-486.328127,-302.734374
6345,6346,3,48777,0,6.141012e+15,-9.823544e+20,-1.295743e+21,-3.013357e+19,7.439512e+19,4.126459e+35,194.730210,-318.359376,-419.921870,-9.765625
6426,6427,3,49720,0,6.225164e+15,-9.823544e+20,-1.295743e+21,-3.013357e+19,6.607856e+19,3.780694e+35,197.398646,-318.359376,-419.921870,-9.765625
6463,6464,3,50021,0,6.255572e+15,-1.126995e+21,-1.428331e+21,2.591487e+20,4.278450e+19,3.619193e+35,198.362896,-365.234368,-462.890617,83.984374
6516,6517,3,50631,0,6.309318e+15,-9.823544e+20,-1.295743e+21,-3.013357e+19,6.607856e+19,3.911548e+35,200.067174,-318.359376,-419.921870,-9.765625
6598,6599,3,51551,0,6.393467e+15,-9.823544e+20,-1.295743e+21,-3.013357e+19,6.607856e+19,3.787675e+35,202.735518,-318.359376,-419.921870,-9.765625
6684,6685,3,52500,0,6.477626e+15,-9.823544e+20,-1.295743e+21,-3.013357e+19,6.351836e+19,4.109211e+35,205.404163,-318.359376,-419.921870,-9.765625


In [ ]:
filtered_data.drop(columns=['posx', 'posy', 'posz', 'time'], inplace=True)
filtered_data.to_csv('SNfeedback_185_200.txt', sep='\t', index=False, encoding='utf-8')

In [18]:
def pc2pix_256(pc):
    return pc * (256 / 1000)

In [20]:
new_posx = pc2pix_256(filtered_data['posx_pc']) + 128
new_posx

6239    42.500000
6254    46.500000
6300    34.500002
6302    43.499998
6345    46.500000
6426    46.500000
6463    34.500002
6516    46.500000
6598    46.500000
6684    46.500000
Name: posx_pc, dtype: float64